# SOSD web UI on the Colab GPU

The same page as https://bachrathyd.github.io/SOSD.jl/webgpu/ — model text, sliders, progressive
brute force, MDBM, periodic orbit of the forced system, zoom, PNG export — but every point is
computed here, on the Colab GPU, by the batched solver of SOSD.jl (CUDA). Coarse levels while
dragging run in Float32; the final level, MDBM and the export in **Float64 with a converged
Krylov–Schur iteration** (relative Ritz residual 1e-10), the periodic orbit by GMRES (1e-10).

1. *Runtime → Change runtime type → GPU* (T4 is enough to start; A100 for large charts).
2. Run the cells in order. The first start installs Julia and compiles the GPU kernels
   (5–10 min on a fresh runtime); a model edited in the page compiles once (~10–30 s).
3. When done: *Runtime → Disconnect and delete runtime* (compute units).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%bash
# Julia (juliaup) + the SOSD.jl `gpu` branch + the web-UI environment (logs in /content/webui_setup.log)
export PATH=/root/.juliaup/bin:$PATH
if ! command -v julia >/dev/null; then
  curl -fsSL https://install.julialang.org | sh -s -- --yes --default-channel 1.12 > /content/juliaup.log 2>&1
fi
if [ -d /content/SOSD.jl/.git ]; then
  cd /content/SOSD.jl && git fetch -q --depth 1 origin gpu && git reset -q --hard FETCH_HEAD
else
  git clone -q -b gpu --depth 1 https://github.com/bachrathyd/SOSD.jl /content/SOSD.jl
fi
cd /content/SOSD.jl && git log -1 --format='code: %h %s'
julia --project=gpu/webui -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()' > /content/webui_setup.log 2>&1
tail -n 3 /content/webui_setup.log
echo "setup done"

In [ ]:
# start the server (compiles the kernels of the first example before it says "ready")
import subprocess, time, os
os.environ['PATH'] = '/root/.juliaup/bin:' + os.environ['PATH']
log = open('/content/webui.log', 'w')
srv = subprocess.Popen(['julia', '-t', 'auto', '--project=gpu/webui', 'gpu/webui/server.jl', '8800'],
                       cwd='/content/SOSD.jl', stdout=log, stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    time.sleep(5)
    txt = open('/content/webui.log', encoding='utf-8', errors='replace').read()
    if 'ready' in txt or srv.poll() is not None or time.time() - t0 > 1800:
        break
    print(f'\r{time.time() - t0:5.0f} s: ' + (txt.strip().splitlines() or ['starting'])[-1][:100], end='')
print()
print(txt[-1500:])

In [ ]:
# the page, computed on this runtime's GPU
from google.colab import output
output.serve_kernel_port_as_iframe(8800, height=1150)

Server log: `!tail -n 30 /content/webui.log`. Stop: `srv.terminate()`. The page detects the server
(`api/info`) and uses it instead of WebGPU; the device name is shown in the header.